In [ ]:
# Import necessary libraries
import pandas as pd  # For data manipulation and analysis
import numpy as np  # For numerical operations
from collections import Counter  # For counting hashable objects
import matplotlib.pyplot as plt  # For plotting
import seaborn as sns  # For advanced data visualization
from urllib.parse import urlparse  # For parsing URLs

# Ensure plots are displayed in the notebook (if applicable)
plt.style.use('seaborn')  # Use seaborn style for matplotlib


In [ ]:
# Import the necessary module to mount Google Drive in Google Colab
from google.colab import drive

# Mount Google Drive to access files stored in it
drive.mount('/content/drive')


In [ ]:
# -----------------------------------------------------------------------
# DATA PATHS — update these to match your local or Google Drive setup
# -----------------------------------------------------------------------

# If running on Google Colab, mount Drive first and set paths like:
#   /content/drive/MyDrive/<your-project-folder>/Result_artifacts/...
# If running locally, set paths to your local dataset files.

datapath_ads = "path/to/combined_ads_dataset.csv"
datapath_articles = "path/to/combined_articles_dataset.csv"
datapath_map = "path/to/folder_number_to_source_url_mapping.csv"
datapath_ee24 = "path/to/ee24_simple-with-dates.csv"

In [ ]:
# Convert string representation of lists in 'Image_Links' column to actual lists
article_df['Image_Links'] = article_df['Image_Links'].apply(ast.literal_eval)

# Calculate the number of images (screenshots) per article and create a new column 'Num_of_screen'
article_df['Num_of_screen'] = article_df['Image_Links'].apply(len)

# Calculate the total number of screenshots across all articles
screenshot_num = article_df['Num_of_screen'].sum()


## Basic Statistics for Advertisement Data

This code calculates key statistics for the advertisement dataset, providing insights into ad distribution, product representation, company diversity, and text length characteristics.


In [ ]:
# Basic Statistics

# Total number of ads
total_ads = len(ads_df)

# Count unique products, excluding NA entries
unique_products = ads_df['Product'].dropna().nunique()

# Count unique companies, excluding NA entries
unique_companies = ads_df['Company'].dropna().nunique()

# Distribution of ad placements
ad_placement_distribution = ads_df['Ad_Placement'].value_counts()

# Calculate ad name length statistics
ad_name_length = ads_df['Ad_Name'].str.len()
average_ad_name_length = ad_name_length.mean()
median_ad_name_length = ad_name_length.median()
max_ad_name_length = ad_name_length.max()

# Calculate ad description length statistics
ad_description_length = ads_df['Ad_Description'].str.len()
average_description_length = ad_description_length.mean()
median_description_length = ad_description_length.median()
max_description_length = ad_description_length.max()

# Get the top 20 products by count
product_distribution = ads_df['Product'].dropna().value_counts().head(20)

# Get the top 20 companies, excluding 'Unknown' and 'Not specified'
company_representation = ads_df['Company'].dropna().value_counts()
company_representation = company_representation[~company_representation.index.isin(['Unknown', 'Not specified'])]
company_representation = company_representation.head(20)


In [ ]:
# Summary of the statistics

# Create a dictionary to store the statistics
statistics = {
    'Total Number of ScreenShots': [screenshot_num],  # Assuming screenshot_num is defined elsewhere
    'Total Number of Ads': [total_ads],
    'Unique Products': [unique_products],
    'Unique Companies': [unique_companies],
    'Average Ad Name Length': [average_ad_name_length],
    'Median Ad Name Length': [median_ad_name_length],
    'Max Ad Name Length': [max_ad_name_length],
    'Average Ad Description Length': [average_description_length],
    'Median Ad Description Length': [median_description_length],
    'Max Ad Description Length': [max_description_length],
}

# Convert the statistics dictionary into a pandas DataFrame for easy visualization
statistics_df = pd.DataFrame(statistics)

# Display the statistics DataFrame
statistics_df

## Visualizing Product Distribution

This code snippet creates a bar chart to display the distribution of the top 20 products based on their frequency in the advertisement dataset.


In [ ]:
# Plot the product distribution
plt.figure(figsize=(8, 6))

# Plotting the top 20 products as a bar chart
product_distribution.plot(kind='bar', color='skyblue')

# Add labels and title for the product distribution plot
plt.xlabel('Products')
plt.ylabel('Frequency')
plt.title('Top 20 Products')

# Rotate the x labels for better readability
plt.xticks(rotation=45, ha='right')

# Adjust layout to prevent label cut-off
plt.tight_layout()

# Save the plot as a PNG image
plt.savefig('Top 20 products.png')

# Show the product distribution plot
plt.show()


## Visualizing Company Representation

This code snippet creates a bar chart to showcase the distribution of the top 20 companies based on their frequency in the advertisement dataset.


In [ ]:
# Plot the company representation
plt.figure(figsize=(8, 6))

# Plotting the top 20 companies as a bar chart
company_representation.plot(kind='bar', color='skyblue')

# Add labels and title for the company representation plot
plt.xlabel('Companies')
plt.ylabel('Frequency')
plt.title('Top 20 Companies')

# Rotate the x labels for better readability
plt.xticks(rotation=45, ha='right')

# Adjust layout to prevent label cut-off
plt.tight_layout()

# Save the plot as a PNG image
plt.savefig('Top_20_company.png')

# Show the company representation plot
plt.show()


# Visualizing Ad Distribution by Sector

This code snippet generates a pie chart showing the distribution of advertisements across the top 20 sectors based on their frequency in the advertisement dataset.



In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

# Load the pre-trained tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("cardiffnlp/twitter-roberta-large-topic-latest")
model = AutoModelForSequenceClassification.from_pretrained("cardiffnlp/twitter-roberta-large-topic-latest")

# Function to generate a prompt using ad details
def generate_prompt(ad_name, product, company, ad_description):
    """
    This function generates a formatted prompt with the given ad details.
    """
    return f"Ad Name: {ad_name}\nProduct: {product}\nCompany: {company}\nAd Description: {ad_description}"

# Initialize the text classification pipeline with the pre-trained model
pipe = pipeline('text-classification', 
                model="cardiffnlp/twitter-roberta-large-topic-latest", 
                return_all_scores=True)


In [ ]:
from tqdm import tqdm

In [ ]:
# ads_df.head()

In [ ]:
# Initialize an empty list to store the tags
tags = []

# Loop through the ads DataFrame and generate predictions
for ad_name, product, company, ad_description in tqdm(
    zip(ads_df['Ad_Name'], ads_df['Product'], ads_df['Company'], ads_df['Ad_Description'])
):
    # Generate the prompt for the current ad
    prompt = generate_prompt(ad_name, product, company, ad_description)
    
    # Get predictions from the pipeline
    predictions = pipe(prompt)[0]
    
    # Get the prediction with the highest score
    pred = max(predictions, key=lambda x: x['score'])
    
    # Append the prediction to the tags list
    tags.append(pred)



In [ ]:
# Extract the sector labels from the tags
sector_list = [tag['label'] for tag in tags]

# Count occurrences of each sector using the Counter class
sector_counts = Counter(sector_list)

# Get the top 10 sectors by count
top_10_sectors = sector_counts.most_common(10)

# Separate sector names and their counts for pie chart plotting
sectors, ad_counts = zip(*top_10_sectors)

# Create and display the pie chart
plt.figure(figsize=(8, 8))
plt.pie(ad_counts, labels=sectors, autopct='%1.1f%%', startangle=90)
plt.title('Top 10 Sectors - Ad Distribution')
plt.axis('equal')  # Equal aspect ratio ensures that the pie is drawn as a circle.
plt.show()


In [ ]:
def merge_ad_article_data(ads_df: pd.DataFrame, articles_df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge ads and articles datasets based on matching image URLs.

    Parameters:
    -----------
    ads_df : DataFrame
        Dataset containing ad information with 'Image_Link' column
    articles_df : DataFrame
        Dataset containing article information with 'Image_Links' column (array/list)

    Returns:
    --------
    DataFrame
        Enhanced ads dataset with article metadata for each ad
    """
    # Create a mapping of image URLs to article metadata
    image_to_article = {}
    for _, article in articles_df.iterrows():
        # Convert string representation of list to actual list if needed
        if isinstance(article['Image_Links'], str):
            image_links = eval(article['Image_Links'])
        else:
            image_links = article['Image_Links']

        # Add article metadata for each image
        for img_url in image_links:
            image_to_article[img_url] = {
                'Published_Date': article['Published_Date'],
                'Sentiment': article['Sentiment'],
                'Tone': article['Tone'],
                'Article_Text': article['Article_Text'],
                'Headline': article['Headline'],
                'Publishing_Agency_Name': article['Publishing_Agency_Name'],
                'Keywords_Tags': article['Keywords_Tags'],
                'Related_Article_Links': article['Related_Article_Links'],
                'Narrative': article['Narrative']
                # Add any other article metadata you need
            }

    # Add article metadata to ads based on matching image URLs
    enhanced_ads = []
    unmatched_ads = []

    for _, ad in ads_df.iterrows():
        ad_data = ad.to_dict()
        if ad['Image_Link'] in image_to_article:
            ad_data.update(image_to_article[ad['Image_Link']])
            enhanced_ads.append(ad_data)
        else:
            unmatched_ads.append(ad['Image_Link'])

    # Create merged dataframe
    merged_df = pd.DataFrame(enhanced_ads)

    # Preprocess dates
    merged_df['Published_Date'] = pd.to_datetime(
        merged_df['Published_Date'],
        format='%d-%m-%Y',
        errors='coerce'
    )

    # Add derived date features
    merged_df['Month'] = merged_df['Published_Date'].dt.month
    merged_df['Month_Name'] = merged_df['Published_Date'].dt.month_name()
    merged_df['Weekday'] = merged_df['Published_Date'].dt.day_name()

    # Print matching statistics
    print(f"Total ads: {len(ads_df)}")
    print(f"Successfully matched ads: {len(enhanced_ads)}")
    print(f"Unmatched ads: {len(unmatched_ads)}")
    if unmatched_ads:
        print("\nSample of unmatched ad images:")
        for url in unmatched_ads[:5]:
            print(f"- {url}")

    return merged_df

### Heatmap for company/products vs Media House

In [ ]:
#combined_original_df = pd.concat([original_df1,original_df2])

In [ ]:
#mdf = merge_ad_article_data(ads_df,article_df)


In [ ]:
"""
# Clean 'ID' column in merged_data to remove '#' character if the value is a string
folder_url_map_df['ID'] = folder_url_map_df['ID'].apply(lambda x: str(x).replace('#', '') if isinstance(x, str) else x)

# Merge the 'mdf' dataframe with 'folder_url_map_df' based on 'Folder_Number' and 'ID' columns
mdf = mdf.merge(folder_url_map_df, left_on='Folder_Number', right_on='ID', how='left')

# Drop 'ID' and 'Observations' columns from the 'mdf' dataframe as they are no longer needed
mdf = mdf.drop(columns=['ID', 'Observations'])

# Filter rows with non-null 'Links' column values
filtered_mdf = mdf.dropna(subset=['Links'])

# Merge 'filtered_mdf' with 'combined_original_df' dataframe based on the 'Links' and 'source_url' columns
merged_data_with_org = filtered_mdf.merge(combined_original_df, left_on='Links', right_on='source_url', how='left')
"""

In [ ]:
"""# Function to extract the domain from a URL
def extract_domain(url):
    try:
        # Use urlparse to extract the domain (netloc) from the URL
        return urlparse(url).netloc
    except:
        # Return None if the URL is invalid or an error occurs
        return None

# Apply the extract_domain function to 'source_url' and create a new column 'media_outlet'
merged_data_with_org['media_outlet'] = merged_data_with_org['source_url'].apply(extract_domain)

merged_data_with_org =merged_data_with_org[~merged_data_with_org['Company'].isin(['Unknown', 'Not specified'])]
# Count occurrences of each combination of 'Company' and 'media_outlet'
company_media_heatmap_data = (
    merged_data_with_org.groupby(['Company', 'media_outlet'])
    .size()
    .unstack(fill_value=0)
)

# Get the top 10 companies based on the total counts across all media outlets
top_10_companies = company_media_heatmap_data.sum(axis=1).nlargest(10).index

# Filter the heatmap data to only include the top 10 companies
company_media_heatmap_data_top_10 = company_media_heatmap_data.loc[top_10_companies]

# Count occurrences of each combination of 'Product' and 'media_outlet'
product_media_heatmap_data = (
    merged_data_with_org.groupby(['Product', 'media_outlet'])
    .size()
    .unstack(fill_value=0)
)

# Get the top 10 products based on the total counts across all media outlets
top_10_products = product_media_heatmap_data.sum(axis=1).nlargest(10).index

# Filter the heatmap data to only include the top 10 products
product_media_heatmap_data_top_10 = product_media_heatmap_data.loc[top_10_products]

# Plot heatmap for companies vs media outlets
plt.figure(figsize=(12, 8))
sns.heatmap(
    company_media_heatmap_data_top_10,
    cmap='coolwarm',
    annot=False,  # Do not annotate the heatmap cells
    fmt='d',      # Format the values as integers
    linewidths=0.5
)
plt.title('Heatmap of Companies vs Media Outlets')
plt.xlabel('Media Outlet')
plt.ylabel('Company')
plt.show()

# Plot heatmap for products vs media outlets
plt.figure(figsize=(12, 8))
sns.heatmap(
    product_media_heatmap_data_top_10,
    cmap='coolwarm',
    annot=False,  # Do not annotate the heatmap cells
    fmt='d',      # Format the values as integers
    linewidths=0.5
)
plt.title('Heatmap of Products vs Media Outlets')
plt.xlabel('Media Outlet')
plt.ylabel('Product')
plt.show()
"""

In [ ]:
unicef_ads_data = mdf[(mdf['Company'].str.contains('UNICEF', case=False, na=False)) |
                      (mdf['Ad_Name'].str.contains('UNICEF', case=False, na=False))]

In [ ]:
unicef_ads_data

In [ ]:
sentiment_counts = unicef_ads_data['Sentiment'].value_counts()

# Plot sentiments as a bar chart
plt.figure(figsize=(8, 5))
sentiment_counts.plot(kind='bar', color=['green', 'orange', 'red'], alpha=0.7)
plt.title('Sentiment Distribution for UNICEF Ads')
plt.xlabel('Sentiment')
plt.ylabel('Number of Articles')
plt.xticks(rotation=0)
plt.show()

In [ ]:
from wordcloud import WordCloud

# Combine all text from 'article_text' and 'headline'
text_data = ' '.join(unicef_ads_data['Article_Text'].fillna('')) + ' ' + ' '.join(unicef_ads_data['Headline'].fillna(''))

# Generate Word Cloud
wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text_data)

# Plot Word Cloud
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Topics in Articles advertising about UNICEF')
plt.show()

# Luis

## Check if it's one screenshot per row

In [ ]:
ads_df.Image_Link.value_counts()

## Map placements to less granular categories

In [ ]:
def mapper(s):
  s = s.lower()
  if 'sidebar' in s:
    return 'sidebar'
  elif 'inline' in s:
    return 'inline'
  elif 'top' in s:
    return 'top section'
  elif 'bottom' in s:
    return 'bottom section'
  elif 'middle' in s:
    return 'middle section'
  elif 'main' in s:
    return 'main section'
  elif 'corner' in s:
    return 'corner'
  elif 'banner' in s:
    return 'banner'
  elif 'popup' in s or 'pop-up' in s or 'pop up' in s:
    return 'popup'
  elif 'overlay' in s:
    return 'overlay'
  elif 'section' in s:
    return 'other section'
  elif 'body' in s:
    return 'article body'
  else:
    return 'other'

In [ ]:
ads_df['Coarse_Ad_Placement'] = ads_df['Ad_Placement'].apply(mapper)

In [ ]:
ads_df.Coarse_Ad_Placement.value_counts()

In [ ]:
ads_df[ads_df.Coarse_Ad_Placement=='other']

## Merge both dfs

In [ ]:
def merge_ad_article_data(ads_df: pd.DataFrame, articles_df: pd.DataFrame) -> pd.DataFrame:
    """
    Merge ads and articles datasets based on matching image URLs.

    Parameters:
    -----------
    ads_df : DataFrame
        Dataset containing ad information with 'Image_Link' column
    articles_df : DataFrame
        Dataset containing article information with 'Image_Links' column (array/list)

    Returns:
    --------
    DataFrame
        Enhanced ads dataset with article metadata for each ad
    """
    # Create a mapping of image URLs to article metadata
    image_to_article = {}
    for _, article in articles_df.iterrows():
        # Convert string representation of list to actual list if needed
        if isinstance(article['Image_Links'], str):
            image_links = eval(article['Image_Links'])
        else:
            image_links = article['Image_Links']

        # Add article metadata for each image
        for img_url in image_links:
            image_to_article[img_url] = {
                'Published_Date': article['Published_Date'],
                'Sentiment': article['Sentiment'],
                'Tone': article['Tone'],
                'Article_Text': article['Article_Text'],
                'Headline': article['Headline'],
                'Publishing_Agency_Name': article['Publishing_Agency_Name'],
                'Keywords_Tags': article['Keywords_Tags'],
                'Related_Article_Links': article['Related_Article_Links'],
                'Narrative': article['Narrative']
                # Add any other article metadata you need
            }

    # Add article metadata to ads based on matching image URLs
    enhanced_ads = []
    unmatched_ads = []

    for _, ad in ads_df.iterrows():
        ad_data = ad.to_dict()
        if ad['Image_Link'] in image_to_article:
            ad_data.update(image_to_article[ad['Image_Link']])
            enhanced_ads.append(ad_data)
        else:
            unmatched_ads.append(ad['Image_Link'])

    # Create merged dataframe
    merged_df = pd.DataFrame(enhanced_ads)

    # Preprocess dates
    merged_df['Published_Date'] = pd.to_datetime(
        merged_df['Published_Date'],
        format='%d-%m-%Y',
        errors='coerce'
    )

    # Add derived date features
    merged_df['Month'] = merged_df['Published_Date'].dt.month
    merged_df['Month_Name'] = merged_df['Published_Date'].dt.month_name()
    merged_df['Weekday'] = merged_df['Published_Date'].dt.day_name()

    # Print matching statistics
    print(f"Total ads: {len(ads_df)}")
    print(f"Successfully matched ads: {len(enhanced_ads)}")
    print(f"Unmatched ads: {len(unmatched_ads)}")
    if unmatched_ads:
        print("\nSample of unmatched ad images:")
        for url in unmatched_ads[:5]:
            print(f"- {url}")

    return merged_df



In [ ]:
mdf = merge_ad_article_data(ads_df,article_df)

In [ ]:
mdf.head(3)

## Embedding analysis

In [ ]:
# Part 1: Embedding Generation
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.manifold import TSNE

def generate_embeddings(df, column_name, perplexity=30, n_iter=1000):
    """Generate embeddings and reduce dimensionality using t-SNE"""
    # Clean data
    df[column_name] = df[column_name].fillna('')
    mask = df[column_name].str.len() > 0
    clean_data = df[mask]

    # Generate embeddings
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(clean_data[column_name].tolist(), show_progress_bar=True)

    # Apply t-SNE
    tsne = TSNE(n_components=2, perplexity=perplexity, n_iter=n_iter, random_state=42)
    embeddings_2d = tsne.fit_transform(embeddings)

    # Create DataFrame with embeddings and metadata
    embed_df = pd.DataFrame({
        'x': embeddings_2d[:, 0],
        'y': embeddings_2d[:, 1],
        'text': clean_data[column_name],
        'tone': clean_data['Tone'],
        'sentiment': clean_data['Sentiment'],
    })

    return embed_df


# Generate embeddings for Products and Companies
product_embeddings = generate_embeddings(mdf, 'Product')
company_embeddings = generate_embeddings(mdf, 'Company')

In [ ]:
company_embeddings.head(3)

In [ ]:
product_embeddings.head(3)

In [ ]:
import plotly.express as px


In [ ]:
# Cell 2: Product Embeddings by Sentiment
fig_product_sentiment = px.scatter(
    product_embeddings,
    x='x',
    y='y',
    color='sentiment',
    title='Products by Sentiment',
    color_discrete_sequence=px.colors.qualitative.Set1,
    hover_data={
        'x': False,
        'y': False,
        'text': True,
        'sentiment': True
    }
)

fig_product_sentiment.update_layout(
    title={
        'text': 'Products Clustered by Sentiment',
        'y':0.95,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=24)
    },
    plot_bgcolor='white',
    width=1200,
    height=800,
    showlegend=True,
    legend=dict(
        title=dict(text='Sentiment Categories', font=dict(size=16)),
        font=dict(size=14),
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=1.02
    )
)

fig_product_sentiment.update_traces(
    marker=dict(size=12, line=dict(width=1, color='DarkSlateGrey')),
    opacity=0.7
)

fig_product_sentiment.update_xaxes(title_text='t-SNE Dimension 1', title_font=dict(size=16), showgrid=True, gridwidth=1, gridcolor='LightGray')
fig_product_sentiment.update_yaxes(title_text='t-SNE Dimension 2', title_font=dict(size=16), showgrid=True, gridwidth=1, gridcolor='LightGray')

fig_product_sentiment.show()
# save fig to html
fig_product_sentiment.write_html("product_sentiment.html")

In [ ]:
# Cell 1: Product Embeddings Visualization
import plotly.express as px

# Create Product visualization by Tone
fig_product_tone = px.scatter(
    product_embeddings,
    x='x',
    y='y',
    color='tone',
    title='Product Embeddings by Tone',
    color_discrete_sequence=px.colors.qualitative.Set3,
    hover_data={
        'x': False,
        'y': False,
        'text': True,
        'tone': True
    }
)

fig_product_tone.update_layout(
    title={
        'text': 'Product Embeddings Clustered by Tone',
        'y':0.95,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=24)
    },
    plot_bgcolor='white',
    width=1200,
    height=800,
    showlegend=True,
    legend=dict(
        title=dict(text='Tone Categories', font=dict(size=16)),
        font=dict(size=14),
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=1.02
    )
)

fig_product_tone.update_traces(
    marker=dict(size=12, line=dict(width=1, color='DarkSlateGrey')),
    opacity=0.7
)

fig_product_tone.update_xaxes(title_text='t-SNE Dimension 1', title_font=dict(size=16), showgrid=True, gridwidth=1, gridcolor='LightGray')
fig_product_tone.update_yaxes(title_text='t-SNE Dimension 2', title_font=dict(size=16), showgrid=True, gridwidth=1, gridcolor='LightGray')

fig_product_tone.show()

In [ ]:
# Cell 2: Product Embeddings by Sentiment
fig_product_sentiment = px.scatter(
    product_embeddings,
    x='x',
    y='y',
    color='sentiment',
    title='Product Embeddings by Sentiment',
    color_discrete_sequence=px.colors.qualitative.Set1,
    hover_data={
        'x': False,
        'y': False,
        'text': True,
        'sentiment': True
    }
)

fig_product_sentiment.update_layout(
    title={
        'text': 'Product Embeddings Clustered by Sentiment',
        'y':0.95,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=24)
    },
    plot_bgcolor='white',
    width=1200,
    height=800,
    showlegend=True,
    legend=dict(
        title=dict(text='Sentiment Categories', font=dict(size=16)),
        font=dict(size=14),
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=1.02
    )
)

fig_product_sentiment.update_traces(
    marker=dict(size=12, line=dict(width=1, color='DarkSlateGrey')),
    opacity=0.7
)

fig_product_sentiment.update_xaxes(title_text='t-SNE Dimension 1', title_font=dict(size=16), showgrid=True, gridwidth=1, gridcolor='LightGray')
fig_product_sentiment.update_yaxes(title_text='t-SNE Dimension 2', title_font=dict(size=16), showgrid=True, gridwidth=1, gridcolor='LightGray')

fig_product_sentiment.show()

In [ ]:
# Cell 3: Company Embeddings by Tone
fig_company_tone = px.scatter(
    company_embeddings,
    x='x',
    y='y',
    color='tone',
    title='Company Embeddings by Tone',
    color_discrete_sequence=px.colors.qualitative.Set3,
    hover_data={
        'x': False,
        'y': False,
        'text': True,
        'tone': True
    }
)

fig_company_tone.update_layout(
    title={
        'text': 'Company Embeddings Clustered by Tone',
        'y':0.95,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=24)
    },
    plot_bgcolor='white',
    width=1200,
    height=800,
    showlegend=True,
    legend=dict(
        title=dict(text='Tone Categories', font=dict(size=16)),
        font=dict(size=14),
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=1.02
    )
)

fig_company_tone.update_traces(
    marker=dict(size=12, line=dict(width=1, color='DarkSlateGrey')),
    opacity=0.7
)

fig_company_tone.update_xaxes(title_text='t-SNE Dimension 1', title_font=dict(size=16), showgrid=True, gridwidth=1, gridcolor='LightGray')
fig_company_tone.update_yaxes(title_text='t-SNE Dimension 2', title_font=dict(size=16), showgrid=True, gridwidth=1, gridcolor='LightGray')

fig_company_tone.show()

In [ ]:
# Cell 4: Company Embeddings by Sentiment
fig_company_sentiment = px.scatter(
    company_embeddings,
    x='x',
    y='y',
    color='sentiment',
    title='Company Embeddings by Sentiment',
    color_discrete_sequence=px.colors.qualitative.Set1,
    hover_data={
        'x': False,
        'y': False,
        'text': True,
        'sentiment': True
    }
)

fig_company_sentiment.update_layout(
    title={
        'text': 'Company Embeddings Clustered by Sentiment',
        'y':0.95,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': dict(size=24)
    },
    plot_bgcolor='white',
    width=1200,
    height=800,
    showlegend=True,
    legend=dict(
        title=dict(text='Sentiment Categories', font=dict(size=16)),
        font=dict(size=14),
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=1.02
    )
)

fig_company_sentiment.update_traces(
    marker=dict(size=12, line=dict(width=1, color='DarkSlateGrey')),
    opacity=0.7
)

fig_company_sentiment.update_xaxes(title_text='t-SNE Dimension 1', title_font=dict(size=16), showgrid=True, gridwidth=1, gridcolor='LightGray')
fig_company_sentiment.update_yaxes(title_text='t-SNE Dimension 2', title_font=dict(size=16), showgrid=True, gridwidth=1, gridcolor='LightGray')

fig_company_sentiment.show()

## Most freq product/company per sentiment/tone

In [ ]:
mdf.Publishing_Agency_Name

In [ ]:
mdf[(mdf.Company.notna()) & (mdf.Company.apply(len) > 3)]

In [ ]:
# Prepare data for each sentiment

mdf_filt = mdf[(mdf.Company.notna()) & (mdf.Company.apply(len) > 3)]
mdf_filt = mdf_filt[~mdf_filt.Company.isin(['[Unknown]', '[Not specified]', 'Unknown'])]
mdf_filt = mdf_filt[~mdf_filt.Company.isin(['Not specified', '[Not specified]', 'Unknown'])]

positive_companies = mdf_filt[mdf_filt['Sentiment'] == 'Positive']['Company'].value_counts().head(10)
negative_companies = mdf_filt[mdf_filt['Sentiment'] == 'Negative']['Company'].value_counts().head(10)
neutral_companies = mdf_filt[mdf_filt['Sentiment'] == 'Neutral']['Company'].value_counts().head(10)

# Create three separate bar plots
import plotly.graph_objects as go

# Plot for Positive Sentiment
fig_positive = go.Figure()
fig_positive.add_trace(go.Bar(
   x=positive_companies.index,
   y=positive_companies.values,
   marker_color='#2ecc71'
))

fig_positive.update_layout(
   title='Top 10 Companies in Positive Sentiment Articles',
   xaxis_title='Company',
   yaxis_title='Frequency',
   width=1000,
   height=600,
   plot_bgcolor='white',
   title_x=0.5,
   title_font_size=20,
   xaxis_tickangle=45
)
fig_positive.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig_positive.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig_positive.show()

# Plot for Negative Sentiment
fig_negative = go.Figure()
fig_negative.add_trace(go.Bar(
   x=negative_companies.index,
   y=negative_companies.values,
   marker_color='#e74c3c'
))

fig_negative.update_layout(
   title='Top 10 Companies in Negative Sentiment Articles',
   xaxis_title='Company',
   yaxis_title='Frequency',
   width=1000,
   height=600,
   plot_bgcolor='white',
   title_x=0.5,
   title_font_size=20,
   xaxis_tickangle=45
)
fig_negative.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig_negative.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig_negative.show()

# Plot for Neutral Sentiment
fig_neutral = go.Figure()
fig_neutral.add_trace(go.Bar(
   x=neutral_companies.index,
   y=neutral_companies.values,
   marker_color='#3498db'
))

fig_neutral.update_layout(
   title='Top 10 Companies in Neutral Sentiment Articles',
   xaxis_title='Company',
   yaxis_title='Frequency',
   width=1000,
   height=600,
   plot_bgcolor='white',
   title_x=0.5,
   title_font_size=20,
   xaxis_tickangle=45
)
fig_neutral.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig_neutral.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig_neutral.show()

In [ ]:
mdf_filt = mdf[(mdf.Product.notna()) & (mdf.Product.apply(len) > 3)]
mdf_filt = mdf_filt[~mdf_filt.Product.isin(['Unknown', 'Not specified'])]

# Prepare data for each sentiment
positive_companies = mdf_filt[mdf_filt['Sentiment'] == 'Positive']['Product'].value_counts().head(10)
negative_companies = mdf_filt[mdf_filt['Sentiment'] == 'Negative']['Product'].value_counts().head(10)
neutral_companies = mdf_filt[mdf_filt['Sentiment'] == 'Neutral']['Product'].value_counts().head(10)

# Create three separate bar plots
import plotly.graph_objects as go

# Plot for Positive Sentiment
fig_positive = go.Figure()
fig_positive.add_trace(go.Bar(
   x=positive_companies.index,
   y=positive_companies.values,
   marker_color='#2ecc71'
))

fig_positive.update_layout(
   title='Top 10 Products in Positive Sentiment Articles',
   xaxis_title='Product',
   yaxis_title='Frequency',
   width=1000,
   height=600,
   plot_bgcolor='white',
   title_x=0.5,
   title_font_size=20,
   xaxis_tickangle=45
)
fig_positive.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig_positive.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig_positive.show()

# Plot for Negative Sentiment
fig_negative = go.Figure()
fig_negative.add_trace(go.Bar(
   x=negative_companies.index,
   y=negative_companies.values,
   marker_color='#e74c3c'
))

fig_negative.update_layout(
   title='Top 10 Products in Negative Sentiment Articles',
   xaxis_title='Product',
   yaxis_title='Frequency',
   width=1000,
   height=600,
   plot_bgcolor='white',
   title_x=0.5,
   title_font_size=20,
   xaxis_tickangle=45
)
fig_negative.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig_negative.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig_negative.show()

# Plot for Neutral Sentiment
fig_neutral = go.Figure()
fig_neutral.add_trace(go.Bar(
   x=neutral_companies.index,
   y=neutral_companies.values,
   marker_color='#3498db'
))

fig_neutral.update_layout(
   title='Top 10 Products in Neutral Sentiment Articles',
   xaxis_title='Product',
   yaxis_title='Frequency',
   width=1000,
   height=600,
   plot_bgcolor='white',
   title_x=0.5,
   title_font_size=20,
   xaxis_tickangle=45
)
fig_neutral.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig_neutral.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')
fig_neutral.show()

## Analyze correlations

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import chi2_contingency
import numpy as np
from collections import defaultdict

# Function to calculate Cramer's V
def cramers_v(confusion_matrix):
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    min_dim = min(confusion_matrix.shape) - 1
    return np.sqrt(chi2 / (n * min_dim))

# Function to analyze categorical relationships
def analyze_categorical_relationships(df, col1, col2):
    contingency = pd.crosstab(df[col1], df[col2])
    chi2, p_value, _, _ = chi2_contingency(contingency)
    cramer = cramers_v(contingency)
    return {
        'chi2': chi2,
        'p_value': p_value,
        'cramers_v': cramer
    }

In [ ]:
# 1. Overall correlation analysis
categorical_cols = ['Product', 'Company', 'Coarse_Ad_Placement', 'Sentiment', 'Tone', 'Weekday']
correlation_results = defaultdict(dict)

for i in range(len(categorical_cols)):
    for j in range(i + 1, len(categorical_cols)):
        col1, col2 = categorical_cols[i], categorical_cols[j]
        results = analyze_categorical_relationships(mdf, col1, col2)
        correlation_results[col1][col2] = results

# Create heatmap of Cramer's V values
cramer_matrix = np.zeros((len(categorical_cols), len(categorical_cols)))
for i in range(len(categorical_cols)):
    for j in range(i + 1, len(categorical_cols)):
        col1, col2 = categorical_cols[i], categorical_cols[j]
        cramer_matrix[i, j] = correlation_results[col1][col2]['cramers_v']
        cramer_matrix[j, i] = cramer_matrix[i, j]

fig = go.Figure(data=go.Heatmap(
    z=cramer_matrix,
    x=categorical_cols,
    y=categorical_cols,
    colorscale='Viridis',
    text=np.round(cramer_matrix, 3),
    texttemplate='%{text}',
    textfont={"size": 10},
    hoverongaps=False
))

fig.update_layout(
    title='Cramer\'s V Correlation Heatmap',
    xaxis_title='Features',
    yaxis_title='Features'
)
fig.show()

# 2. Analysis for Tone and Sentiment
def analyze_category_distribution(df, target_col, analysis_cols=['Product', 'Company', 'Coarse_Ad_Placement']):
    results = []

    for analysis_col in analysis_cols:
        # Get top 10 categories for each tone/sentiment
        distribution = df.groupby([target_col, analysis_col]).size().reset_index(name='count')
        top_categories = (distribution.sort_values('count', ascending=False)
                         .groupby(target_col)
                         .head(10))

        # Create bar plot
        fig = px.bar(
            top_categories,
            x=analysis_col,
            y='count',
            color=target_col,
            title=f'Top Categories by {target_col}',
            barmode='group'
        )

        fig.update_layout(
            xaxis_tickangle=-45,
            xaxis_title=analysis_col,
            yaxis_title='Count'
        )
        fig.show()

        # Statistical testing
        contingency = pd.crosstab(df[target_col], df[analysis_col])
        chi2, p_value, _, _ = chi2_contingency(contingency)
        cramer = cramers_v(contingency)

        results.append({
            'analysis_col': analysis_col,
            'target_col': target_col,
            'chi2': chi2,
            'p_value': p_value,
            'cramers_v': cramer
        })

    return pd.DataFrame(results)

# Analyze for both Tone and Sentiment
tone_results = analyze_category_distribution(mdf, 'Tone')
sentiment_results = analyze_category_distribution(mdf, 'Sentiment')

# Print statistical results
print("\nTone Analysis Results:")
print(tone_results)
print("\nSentiment Analysis Results:")
print(sentiment_results)

# 3. Time-based analysis
weekday_distribution = mdf.groupby(['Weekday', 'Tone', 'Sentiment']).size().reset_index(name='count')

# Create weekday distribution plot
fig = px.bar(
    weekday_distribution,
    x='Weekday',
    y='count',
    color='Tone',
    facet_col='Sentiment',
    title='Distribution of Tone and Sentiment by Weekday',
    barmode='group'
)

fig.update_layout(
    xaxis_tickangle=-45
)
fig.show()